In [ ]:
#Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile


In [4]:
#File path to the zipped datafile
zip_path = '../../Data/Processed/Data_Compressed.zip'

#List of filenames to read from the zip file
filenames = [
    'Data_Compressed/all_normalized_features.csv',
    'Data_Compressed/kdd_all_normalized_features.csv',
    'Data_Compressed/kdd_expanded_all_scaled.csv',
    'Data_Compressed/kdd_merged_normalized_all.csv'
]

#Corresponding names for the DataFrames
df_names = [
    'xuetangx_df',
    'kdd_df',
    'kdd_expanded_df',
    'kdd_merged_df'
]

#Create an empty dictionary to store the DataFrames
dataframes = {}

#Open the zip file
z = zipfile.ZipFile(zip_path, 'r')

#Loop through the filenames and read them into DataFrames
for i, file in enumerate(filenames):
    print(f"Reading file: {file} from {zip_path}")
    
    #Read each CSV file directly from the zip
    f = z.open(file)
    dataframes[df_names[i]] = pd.read_csv(f)
    f.close()
    
    print(f"{df_names[i]} loaded with shape: {dataframes[df_names[i]].shape}\n")

#Close the zip file after reading
z.close()

#Assign individual DataFrames to variables
xuetangx_df = dataframes['xuetangx_df']
kdd_df = dataframes['kdd_df']
kdd_expanded_df = dataframes['kdd_expanded_df']
kdd_merged_df = dataframes['kdd_merged_df']


Reading file: Data_Compressed/all_normalized_features.csv from ../../Data/Processed/Data_Compressed.zip
xuetangx_df loaded with shape: (225642, 30)

Reading file: Data_Compressed/kdd_all_normalized_features.csv from ../../Data/Processed/Data_Compressed.zip
kdd_df loaded with shape: (200904, 17)

Reading file: Data_Compressed/kdd_expanded_all_scaled.csv from ../../Data/Processed/Data_Compressed.zip
kdd_expanded_df loaded with shape: (120542, 142)

Reading file: Data_Compressed/kdd_merged_normalized_all.csv from ../../Data/Processed/Data_Compressed.zip
kdd_merged_df loaded with shape: (120542, 158)



In [ ]:
#!pip install tensorflow
#!pip install fastai

In [7]:
#Import required libraries for Deep Learning
#Keras DNN Classifier
from keras.models import Sequential
from keras.layers import BatchNormalization, Dense, Dropout
from keras.regularizers import l2
from keras.utils import to_categorical, normalize
from keras import backend as K

#FastAI DL Classifier
from fastai.tabular.all import *

#Metrics for evaluation
from sklearn.metrics import balanced_accuracy_score, accuracy_score, precision_score, recall_score, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split

### Data Preprocessing

In [ ]:
#Display the info of the xuetangx_df DataFrame
xuetangx_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225642 entries, 0 to 225641
Data columns (total 30 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   enroll_id                      225642 non-null  int64  
 1   action_count                   225642 non-null  float64
 2   seek_video_count               225642 non-null  float64
 3   play_video_count               225642 non-null  float64
 4   pause_video_count              225642 non-null  float64
 5   stop_video_count               225642 non-null  float64
 6   load_video_count               225642 non-null  float64
 7   problem_get_count              225642 non-null  float64
 8   problem_check_count            225642 non-null  float64
 9   problem_save_count             225642 non-null  float64
 10  reset_problem_count            225642 non-null  float64
 11  problem_check_correct_count    225642 non-null  float64
 12  problem_check_incorrect_count 

In [11]:
#Display the info of the kdd_df DataFrame
kdd_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200904 entries, 0 to 200903
Data columns (total 17 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   enrollment_id             200904 non-null  int64  
 1   action_count              200904 non-null  float64
 2   server_navigate_count     200904 non-null  float64
 3   server_access_count       200904 non-null  float64
 4   server_problem_count      200904 non-null  float64
 5   server_page_close_count   200904 non-null  float64
 6   server_video_count        200904 non-null  float64
 7   server_discussion_count   200904 non-null  float64
 8   server_wiki_count         200904 non-null  float64
 9   browser_navigate_count    200904 non-null  float64
 10  browser_access_count      200904 non-null  float64
 11  browser_problem_count     200904 non-null  float64
 12  browser_page_close_count  200904 non-null  float64
 13  browser_video_count       200904 non-null  f

In [ ]:
#Display the info of the kdd_expanded_df DataFrame
kdd_expanded_df.columns

Index(['Unnamed: 0', 'enrollment_id', 'truth', 'avg_chapter_delays',
       'server_discussion_percent', 'act_cnt_weekDay_01',
       'browser_html_percent', 'parallel_enrollments', 'browser_dictation',
       'act_cnt_day_00',
       ...
       'server_course_percent', 'browser_course_info_percent',
       'browser_course', 'browser_vertical_percent', 'sessions_in_week_1',
       'sessions_in_week_0', 'sessions_in_week_3', 'sessions_in_week_2',
       'sessions_in_week_4', 'browser_about'],
      dtype='object', length=142)


In [21]:
#Isolate the X and y features for the xuetangx dataset
xuetangx_X = xuetangx_df.drop(columns=['truth'])
xuetangx_X = xuetangx_X.drop(columns=['enroll_id'])
xuetangx_y = xuetangx_df['truth']
#Isolate the X and y features for the kdd dataset
kdd_X = kdd_df.drop(columns=['truth'])
kdd_X = kdd_X.drop(columns=['enrollment_id'])
kdd_y = kdd_df['truth']
#Isolate the X and y features for the kdd_expanded 
kdd_expanded_X = kdd_expanded_df.drop(columns=['truth'])
kdd_expanded_X = kdd_expanded_X.drop(columns=['enrollment_id'])
kdd_expanded_X = kdd_expanded_X.drop(columns=['Unnamed: 0'])
kdd_expanded_y = kdd_expanded_df['truth']

In [23]:
#Split the xuetangx dataset into training and testing sets
xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test = train_test_split(xuetangx_X, xuetangx_y, test_size=0.2, random_state=100)

#Split the kdd dataset into training and testing sets
kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test = train_test_split(kdd_X, kdd_y, test_size=0.2, random_state=100)

#Split the kdd_expanded dataset into training and testing sets
kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test = train_test_split(kdd_expanded_X, kdd_expanded_y, test_size=0.2, random_state=100)

### Keras-TensorFlow

#### Xuetangx

In [ ]:
#Modfy the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test

In [29]:
#Create a DNN model using Keras
dnn_keras = Sequential(layers=[
                            Dense(128, kernel_regularizer=l2(0.001), activation='relu',input_shape=(len(X_train.columns),)),
                            BatchNormalization(),
                            Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
                            BatchNormalization(),
                            Dense(y_train.nunique(), activation='softmax')
])
dnn_keras.compile(
    optimizer='adam', 
    loss='binary_crossentropy')

dnn_keras.fit(X_train, pd.get_dummies(y_train), epochs=100, verbose=0, batch_size=512)
#loss, acc = dnn_keras.evaluate(X_test, pd.get_dummies(y_test), verbose=0)

C:\Users\chanc\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [30]:
#Create predicted values for the test set
y_pred = np.argmax(dnn_keras.predict(X_test), axis=1)
acc = accuracy_score(y_test, y_pred) * 100
bal_acc = balanced_accuracy_score(y_test, y_pred) * 100
rec = recall_score(y_test, y_pred, average='weighted') * 100
prec = precision_score(y_test, y_pred, average='weighted') * 100
auc = roc_auc_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred) * 100

#Store the results in a dictionary
results = {
    'Accuracy': acc,
    'Balanced Accuracy': bal_acc,
    'Recall': rec,
    'Precision': prec,
    'AUC': auc,
    'F1 Score': f1
}
#Convert the results dictionary to a DataFrame
xuetang_results_df = pd.DataFrame(list(results.items()), columns=['Metric', 'Value'])
#Display the results DataFrame
print(xuetang_results_df)

1411/1411 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
              Metric      Value
0           Accuracy  84.105564
1  Balanced Accuracy  72.130012
2             Recall  84.105564
3          Precision  83.297537
4                AUC  72.130012
5           F1 Score  90.100609


#### KDD (Experiment 1)

In [31]:
#Modfy the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test

In [32]:
#Create a DNN model using Keras
dnn_keras = Sequential(layers=[
                            Dense(128, kernel_regularizer=l2(0.001), activation='relu',input_shape=(len(X_train.columns),)),
                            BatchNormalization(),
                            Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
                            BatchNormalization(),
                            Dense(y_train.nunique(), activation='softmax')
])
dnn_keras.compile(
    optimizer='adam', 
    loss='binary_crossentropy')

dnn_keras.fit(X_train, pd.get_dummies(y_train), epochs=100, verbose=0, batch_size=512)
#loss, acc = dnn_keras.evaluate(X_test, pd.get_dummies(y_test), verbose=0)

C:\Users\chanc\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [33]:
#Create predicted values for the test set
y_pred = np.argmax(dnn_keras.predict(X_test), axis=1)
acc = accuracy_score(y_test, y_pred) * 100
bal_acc = balanced_accuracy_score(y_test, y_pred) * 100
rec = recall_score(y_test, y_pred, average='weighted') * 100
prec = precision_score(y_test, y_pred, average='weighted') * 100
auc = roc_auc_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred) * 100

#Store the results in a dictionary
results = {
    'Accuracy': acc,
    'Balanced Accuracy': bal_acc,
    'Recall': rec,
    'Precision': prec,
    'AUC': auc,
    'F1 Score': f1
}
#Convert the results dictionary to a DataFrame
kdd_results_df = pd.DataFrame(list(results.items()), columns=['Metric', 'Value'])
#Display the results DataFrame
print(kdd_results_df)

1256/1256 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
              Metric      Value
0           Accuracy  86.065553
1  Balanced Accuracy  71.940144
2             Recall  86.065553
3          Precision  85.186784
4                AUC  71.940144
5           F1 Score  91.617385


#### KDD (Experiment 2)

In [34]:
#Modfy the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test

In [35]:
#Create a DNN model using Keras
dnn_keras = Sequential(layers=[
                            Dense(128, kernel_regularizer=l2(0.001), activation='relu',input_shape=(len(X_train.columns),)),
                            BatchNormalization(),
                            Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
                            BatchNormalization(),
                            Dense(y_train.nunique(), activation='softmax')
])
dnn_keras.compile(
    optimizer='adam', 
    loss='binary_crossentropy')

dnn_keras.fit(X_train, pd.get_dummies(y_train), epochs=100, verbose=0, batch_size=512)
#loss, acc = dnn_keras.evaluate(X_test, pd.get_dummies(y_test), verbose=0)

C:\Users\chanc\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
#Create predicted values for the test set
y_pred = np.argmax(dnn_keras.predict(X_test), axis=1)
acc = accuracy_score(y_test, y_pred) * 100
bal_acc = balanced_accuracy_score(y_test, y_pred) * 100
rec = recall_score(y_test, y_pred, average='weighted') * 100
prec = precision_score(y_test, y_pred, average='weighted') * 100
auc = roc_auc_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred) * 100

#Store the results in a dictionary
results = {
    'Accuracy': acc,
    'Balanced Accuracy': bal_acc,
    'Recall': rec,
    'Precision': prec,
    'AUC': auc,
    'F1 Score': f1
}
#Convert the results dictionary to a DataFrame
kdd_expanded_results_df = pd.DataFrame(list(results.items()), columns=['Metric', 'Value'])
#Display the results DataFrame
print(kdd_expanded_results_df)

754/754 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
              Metric      Value
0           Accuracy  85.810278
1  Balanced Accuracy  73.650539
2             Recall  85.810278
3          Precision  84.890963
4                AUC  73.650539
5           F1 Score  91.341213


### FastAI

#### Xuetangx

In [38]:
#Modfy the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test

In [40]:
#Create a DNN model using FastAI
splits = RandomSplitter(valid_pct=0.2)(range_of(X_train))
feature_set = X_train.columns
y_names = ['truth']
#print(df[:5])
tp = TabularPandas(xuetangx_df, procs=[],
                    cat_names= [],
                    cont_names = list(feature_set),
                    y_names= y_names,
                    splits=splits)

dls = tp.dataloaders(bs=64)
#dls.show_batch()
#return
dnn_fastai = tabular_learner(dls, metrics=accuracy)
dnn_fastai.fit_one_cycle(5)

Training and Evaluating Fast.ai...


epoch,train_loss,valid_loss,accuracy,time
0,0.131837,0.137048,0.239017,00:26
1,0.130642,0.132157,0.239017,00:27
2,0.127453,0.128317,0.239017,00:31
3,0.120849,0.126150,0.239017,00:32
4,0.118240,0.125980,0.239017,00:36


In [ ]:
y_pred = []
#print('Length of test set: {}'.format(len(y_test)))
for j in range(len(y_test)):
    row, clas, probs = dnn_fastai.predict(X_test.iloc[j])
    #print(clas)
    pred = 0
    if clas >= tensor(0.5):
        pred = 1
    y_pred.append(pred)

acc = accuracy_score(y_test, y_pred) * 100
bal_acc = balanced_accuracy_score(y_test, y_pred) * 100
rec = recall_score(y_test, y_pred, average='weighted') * 100
prec = precision_score(y_test, y_pred, average='weighted') * 100
auc = roc_auc_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred) * 100

#Store the results in a dictionary
results = {
    'Accuracy': acc,
    'Balanced Accuracy': bal_acc,
    'Recall': rec,
    'Precision': prec,
    'AUC': auc,
    'F1 Score': f1
}
#Convert the results dictionary to a DataFrame
xuetangx_fastai_results_df = pd.DataFrame(list(results.items()), columns=['Metric', 'Value'])
#Display the results DataFrame
print(xuetangx_fastai_results_df)

#### KDD (Experiment 1)

In [ ]:
#Modfy the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test

In [ ]:
#Create a DNN model using FastAI
splits = RandomSplitter(valid_pct=0.2)(range_of(X_train))
feature_set = X_train.columns
y_names = ['truth']
#print(df[:5])
tp = TabularPandas(xuetangx_df, procs=[],
                    cat_names= [],
                    cont_names = list(feature_set),
                    y_names= y_names,
                    splits=splits)

dls = tp.dataloaders(bs=64)
#dls.show_batch()
#return
dnn_fastai = tabular_learner(dls, metrics=accuracy)
dnn_fastai.fit_one_cycle(5)

In [ ]:
y_pred = []
#print('Length of test set: {}'.format(len(y_test)))
for j in range(len(y_test)):
    row, clas, probs = dnn_fastai.predict(X_test.iloc[j])
    #print(clas)
    pred = 0
    if clas >= tensor(0.5):
        pred = 1
    y_pred.append(pred)

acc = accuracy_score(y_test, y_pred) * 100
bal_acc = balanced_accuracy_score(y_test, y_pred) * 100
rec = recall_score(y_test, y_pred, average='weighted') * 100
prec = precision_score(y_test, y_pred, average='weighted') * 100
auc = roc_auc_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred) * 100

#Store the results in a dictionary
results = {
    'Accuracy': acc,
    'Balanced Accuracy': bal_acc,
    'Recall': rec,
    'Precision': prec,
    'AUC': auc,
    'F1 Score': f1
}
#Convert the results dictionary to a DataFrame
kdd_fastai_results_df = pd.DataFrame(list(results.items()), columns=['Metric', 'Value'])
#Display the results DataFrame
print(kdd_fastai_results_df)

#### KDD (Experiment 2)

In [ ]:
#Modfy the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test

In [ ]:
#Create a DNN model using FastAI
splits = RandomSplitter(valid_pct=0.2)(range_of(X_train))
feature_set = X_train.columns
y_names = ['truth']
#print(df[:5])
tp = TabularPandas(xuetangx_df, procs=[],
                    cat_names= [],
                    cont_names = list(feature_set),
                    y_names= y_names,
                    splits=splits)

dls = tp.dataloaders(bs=64)
#dls.show_batch()
#return
dnn_fastai = tabular_learner(dls, metrics=accuracy)
dnn_fastai.fit_one_cycle(5)

In [ ]:
y_pred = []
#print('Length of test set: {}'.format(len(y_test)))
for j in range(len(y_test)):
    row, clas, probs = dnn_fastai.predict(X_test.iloc[j])
    #print(clas)
    pred = 0
    if clas >= tensor(0.5):
        pred = 1
    y_pred.append(pred)

acc = accuracy_score(y_test, y_pred) * 100
bal_acc = balanced_accuracy_score(y_test, y_pred) * 100
rec = recall_score(y_test, y_pred, average='weighted') * 100
prec = precision_score(y_test, y_pred, average='weighted') * 100
auc = roc_auc_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred) * 100

#Store the results in a dictionary
results = {
    'Accuracy': acc,
    'Balanced Accuracy': bal_acc,
    'Recall': rec,
    'Precision': prec,
    'AUC': auc,
    'F1 Score': f1
}
#Convert the results dictionary to a DataFrame
kdd_expanded_fastai_results_df = pd.DataFrame(list(results.items()), columns=['Metric', 'Value'])
#Display the results DataFrame
print(kdd_expanded_fastai_results_df)